# Kubernetes GitOps with Gitea {background-color="black" background-image="https://images.unsplash.com/photo-1618401471353-b98afee0b2eb?w=1600" background-size="cover" background-opacity="0.4"}

# The Course Capstone {background-color="#1a1a2e" background-image="https://images.unsplash.com/photo-1558494949-ef010cbdcc31?w=1600" background-size="cover" background-opacity="0.3"}

## Everything Converges Here

```{mermaid}
timeline
    title Cloud Systems Lab 2026
    Lesson 1-2 : Docker fundamentals
             : Compose
    Lesson 3-4 : Docker Swarm
              : Gitea CI/CD for Swarm
    Lesson 5-6 : IaC with OpenTofu
              : Multipass VMs
              : AWS CloudFormation
    Lesson 7   : Gitea + act_runner (shell)
              : Automated Swarm provisioning
    Lesson 8   : GitHub Actions + AWS OIDC
    Lesson 9   : Kubernetes with kubeadm
              : Multipass cluster
    Lesson 10  : GitOps for K8s
              : infra + app pipelines
```

# Recap {background-color="white" background-image="https://images.unsplash.com/photo-1497633762265-9d179a990aa6?q=80&w=2073&" background-size="cover" background-opacity="0.7"}

## What We Built in Last Lesson

::::{.columns}

::: {.fragment .column width="50%"}

**The cluster**

```bash
# 1 — Provision VMs
tofu apply

# 2 — Configure all nodes
ansible-playbook -i terraform/hosts.ini \
  ansible/00-prerequisites.yml

# 3 — Bootstrap control plane
ansible-playbook -i terraform/hosts.ini \
  ansible/01-control-plane.yml

# 4 — Join workers
ansible-playbook -i terraform/hosts.ini \
  ansible/02-workers.yml
```

:::

::: {.fragment .column width="50%"}

**Result after ~10 minutes**

```
NAME              STATUS   ROLES           AGE
control-plane-1   Ready    control-plane   8m
worker-1          Ready    <none>          5m
worker-2          Ready    <none>          5m
```

We ran all 4 commands manually from the terminal.

::: {.callout-warning}
**Same old problem:** not reproducible, requires your laptop, no audit trail, credentials on disk.
:::

:::

::::

## The Right Tool for Each Layer

::::{.columns}

::: {.fragment .column width="40%"}

```{mermaid}
flowchart TD
    INFRA["🏗️ Infrastructure Layer<br/>Multipass VMs · SSH keys<br/>Ansible inventory"]:::infra
    OS["🖥️ OS / Node Layer<br/>swap · kernel modules<br/>containerd · sysctl"]:::os
    K8S["☸️ Kubernetes Layer<br/>kubeadm init · kubeadm join<br/>etcd · kube-apiserver<br/>kubelet · Flannel CNI"]:::k8s
    APP["📦 Workload Layer<br/>Deployments · Services<br/>ConfigMaps · Scaling<br/>Rolling updates"]:::app

    INFRA -->|"VMs running<br/>inventory written"| OS
    OS -->|"nodes prepared"| K8S
    K8S -->|"kubectl get nodes ✅"| APP

    classDef infra fill:#7B68EE,color:#fff,stroke:#4b3fa0
    classDef os   fill:#E74C3C,color:#fff,stroke:#a93226
    classDef k8s  fill:#326CE5,color:#fff,stroke:#1a4a9f
    classDef app  fill:#27AE60,color:#fff,stroke:#1a7a42
```

:::

::: {.fragment .column width="60%"}

| Layer | Tool | Responsibility |
|---|---|---|
| 🏗️ Infrastructure | **OpenTofu** | VMs, SSH keys, Ansible inventory |
| 🖥️ OS / Node | **Ansible** | swap, modules, containerd, kubelet |
| ☸️ Kubernetes | **kubeadm** (via Ansible) | cluster bootstrap, TLS, etcd, CNI |
| 📦 Workloads | **kubectl** | Deployments, Services, rollouts |

::: {.fragment}
*"OpenTofu gives Ansible machines to configure. Ansible gives kubectl a cluster to deploy into."*
:::

::: {.callout-important .fragment}
**Why not `ansible.kubernetes.core.k8s`?**  
It wraps `kubectl apply` — you'd be duplicating Kubernetes's own reconciliation engine. The K8s API *is* the operator. Ansible adds no value past the cluster boundary.
:::

:::

::::


# Pipeline Design {background-color="#2d1b69" background-image="https://images.unsplash.com/photo-1556761175-4b46a572b786?w=1600" background-size="cover" background-opacity="0.3"}

## One Repo fits All

::: {.callout-note title="All in One"}
The repo contains *both* cluster infrastructure (`terraform/`, `ansible/`) *and* application manifests (`k8s/`). In production you'd split these into two repos — one for infra, one per app.
:::

```{mermaid}
flowchart LR
    TF["📁 terraform/**"]:::tofu
    AN["📁 ansible/**"]:::ansible
    K8S["📁 k8s/**"]:::kubectl

    P1["provision<br/>tofu apply"]:::tofu
    P2["configure<br/>00/01/02-*.yml"]:::ansible
    P3["save kubeconfig<br/>~/k8s-config"]:::shell

    D1["kubectl apply -f k8s/"]:::kubectl
    D2["kubectl rollout status"]:::kubectl

    TF & AN -->|"push triggers infra.yml"| P1
    P1 --> P2 --> P3

    K8S -->|"push triggers deploy.yml"| D1
    D1 --> D2

    P3 -.->|"~/k8s-config<br/>persists on host"| D1

    classDef tofu    fill:#7B68EE,color:#fff,stroke:#4b3fa0
    classDef ansible fill:#E74C3C,color:#fff,stroke:#a93226
    classDef shell   fill:#555,color:#fff,stroke:#333
    classDef kubectl fill:#326CE5,color:#fff,stroke:#1a4a9f
```


## Repository Structure

::::{.columns}

::: {.column width="50%"}

```
k8s-gitea-cicd/
├── .gitea/
│   └── workflows/
│       ├── infra.yml      ← cluster lifecycle
│       ├── deploy.yml     ← workload deploy
│       └── destroy.yml    ← manual teardown
├── terraform/             ← same as Lesson 9
│   ├── main.tf            (state → ~/k8s-terraform.tfstate)
│   ├── variables.tf
│   └── inventory.tpl      (inventory → ~/k8s-hosts.ini)
├── ansible/               ← same playbooks
│   ├── 00-prerequisites.yml
│   ├── 01-control-plane.yml
│   └── 02-workers.yml
├── k8s/
│   ├── nginx-deployment.yaml
│   └── nginx-service.yaml
└── setup-repo.sh
```

:::

::: {.fragment .column width="50%"}

**Why separate triggers?**

```yaml
# infra.yml — only runs when cluster code changes
on:
  push:
    paths: ['terraform/**', 'ansible/**']

# deploy.yml — only runs when app manifests change
on:
  push:
    paths: ['k8s/**']
```

::: {.callout-tip}
Changing your `nginx-deployment.yaml` does **not** re-provision the cluster. Changing `variables.tf` does **not** redeploy your app.
:::

:::

::::

## Prerequisites — Gitea & act_runner

::::{.columns}

::: {.fragment .column width="50%"}

**Start Gitea** (`lab/gitea-setup/`)

```bash
cd lab/gitea-setup

# First time only — downloads binary + opens wizard
./install-gitea.sh
# → http://localhost:3000  (complete setup wizard)

# Subsequent runs — just start the server
./gitea web --config conf/app.ini --port 3000 &
```

:::

::: {.fragment .column width="50%"}

**Register & start act_runner** (`lab/gitea-setup/`)

```bash
cd lab/gitea-setup

# First time only — downloads binary + registers runner
# Get a runner token at:
# http://localhost:3000/-/admin/runners → Create Runner
GITEA_TOKEN=<runner-token> ./register-runner.sh
# labels: self-hosted,linux,multipass

# Start the runner (keep this terminal open)
./act_runner daemon
```

:::

::::

::: {.fragment}

**Lab: Bootstrap the repo and start the pipeline**

```bash 
cd lab/k8s-gitea-cicd

# Create Gitea repo + enable Actions + upload secrets
GITEA_TOKEN=<your-pat> ./setup-repo.sh

# Copy to a clean dir OUTSIDE this repo, excluding k8s/ for now
# (deploy.yml would fail — the cluster isn't ready yet)
rsync -a --exclude='.git' --exclude='k8s' . ~/k8s-infra/
cd ~/k8s-infra
git init && git branch -M main
git remote add gitea http://localhost:3000/<user>/k8s-infra.git
git add . && git commit -m "feat: provision k8s cluster"
git push gitea main
# → only infra.yml triggers. We'll add k8s/ once the cluster is Ready.
```

:::


# State & Secrets {background-color="#1a2a1a" background-image="https://images.unsplash.com/photo-1614064641938-3bbee52942c7?w=1600" background-size="cover" background-opacity="0.3"}

## Host Runner — Persistent State


Act runner executes jobs directly on your laptop:

| File | Written by | Used by |
|---|---|---|
| `~/k8s-terraform.tfstate` | `tofu apply` | `tofu plan`, `tofu destroy` | 
| `~/cloud-init.rendered.yml` | `local_file.cloud_init` | Multipass VM creation |
| `~/k8s-hosts.ini` | `local_file.inventory` | all 3 Ansible playbooks | | 
| `/tmp/k8s_join_command.sh` | `01-control-plane.yml` | `02-workers.yml` | 
| `~/k8s-config` | infra configure job | deploy.yml | 




## Secrets Setup

Only two secrets are needed — the same pattern as Lesson 7:

| Secret | Value | Used in |
|---|---|---|
| `SSH_PRIVATE_KEY` | Content of `id_ed25519` | All pipeline jobs |
| `SSH_PUBLIC_KEY` | Content of `id_ed25519.pub` | `provision` (cloud-init) |

Secrets are created by `setup-repo.sh` — no manual steps needed.

Go in <http://localhost:3000/admin/k8s-infra/settings/actions/secrets> to verify.


::: {.callout-tip .fragment}
No KUBECONFIG secret needed — `~/k8s-config` is written directly on the host by the infra pipeline and reused by deploy.
:::

# `infra.yml` — Cluster Lifecycle {background-color="#4b3fa0" background-image="https://images.unsplash.com/photo-1451187580459-43490279c0fa?w=1600" background-size="cover" background-opacity="0.3"}

## Job 1: `provision` — OpenTofu

```yaml
jobs:
  provision:
    runs-on: self-hosted
    steps:
      - uses: actions/checkout@v4

      - name: Write SSH key pair
        run: |
          echo "${{ secrets.SSH_PRIVATE_KEY }}" > terraform/id_ed25519
          echo "${{ secrets.SSH_PUBLIC_KEY }}"  > terraform/id_ed25519.pub
          chmod 600 terraform/id_ed25519

      - name: tofu init
        run: tofu -chdir=terraform init

      - name: tofu plan
        run: tofu -chdir=terraform plan -out=tfplan

      - name: tofu apply
        run: tofu -chdir=terraform apply -auto-approve tfplan
```

::: {.callout-note .fragment}
State is at `~/k8s-terraform.tfstate` — `tofu plan` will show **no changes** if the cluster already exists (idempotent).
:::

## Job 2: `configure` — Ansible + kubeconfig

```yaml
  configure:
    runs-on: self-hosted
    needs: provision          # waits for VMs to exist
    steps:
      - uses: actions/checkout@v4

      - name: Write SSH private key
        run: |
          echo "${{ secrets.SSH_PRIVATE_KEY }}" > terraform/id_ed25519
          chmod 600 terraform/id_ed25519

      - name: Wait for VMs to be SSH-ready
        run: sleep 30

      - name: Install K8s prerequisites on all nodes
        run: ansible-playbook -i ~/k8s-hosts.ini ansible/00-prerequisites.yml

      - name: Bootstrap control plane
        run: ansible-playbook -i ~/k8s-hosts.ini ansible/01-control-plane.yml

      - name: Join worker nodes
        run: ansible-playbook -i ~/k8s-hosts.ini ansible/02-workers.yml

      - name: Save kubeconfig to host runner
        run: |
          CONTROL_PLANE_IP=$(grep ansible_host ~/k8s-hosts.ini \
            | head -1 | grep -oP 'ansible_host=\K\S+')
          scp -i terraform/id_ed25519 -o StrictHostKeyChecking=no \
            ubuntu@${CONTROL_PLANE_IP}:/home/ubuntu/.kube/config \
            ~/k8s-config
          sed -i "s|https://[0-9.]*:6443|https://${CONTROL_PLANE_IP}:6443|" \
            ~/k8s-config
          chmod 600 ~/k8s-config
          KUBECONFIG=~/k8s-config kubectl get nodes
```



## ✅ Verify the cluster is Ready

```bash
KUBECONFIG=~/k8s-config kubectl get nodes
# NAME              STATUS   ROLES           AGE
# control-plane-1   Ready    control-plane   ~8m
# worker-1          Ready    <none>          ~5m
# worker-2          Ready    <none>          ~5m
```


# `deploy.yml` — Workload Pipeline {background-color="#1a3a5c" background-image="https://images.unsplash.com/photo-1558494949-ef010cbdcc31?w=1600" background-size="cover" background-opacity="0.3"}

## The K8s Manifests

::::{.columns}

::: {.column width="50%"}

```yaml
# nginx-deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: nginx
spec:
  replicas: 3
  selector:
    matchLabels: {app: nginx}
  template:
    metadata:
      labels: {app: nginx}
    spec:
      containers:
        - name: nginx
          image: nginx:1.27-alpine
```

:::

::: {.fragment .column width="50%"}

```yaml
# nginx-service.yaml
apiVersion: v1
kind: Service
metadata:
  name: nginx
spec:
  type: NodePort
  selector: {app: nginx}
  ports:
    - port: 80
      targetPort: 80
      nodePort: 30080
```

::: {.callout-tip}
`NodePort 30080` makes the app reachable on every worker node at `http://<worker-ip>:30080` — no ingress controller needed for this lab.
:::

:::

::::


## `deploy.yml` — The Pipeline

```yaml
on:
  push:
    branches: [main]
    paths: ['k8s/**']
  workflow_dispatch:

jobs:
  deploy:
    runs-on: self-hosted
    steps:
      - uses: actions/checkout@v4

      - name: Verify cluster is reachable
        run: KUBECONFIG=~/k8s-config kubectl get nodes

      - name: Apply manifests
        run: KUBECONFIG=~/k8s-config kubectl apply -f k8s/

      - name: Wait for rollout
        run: |
          KUBECONFIG=~/k8s-config \
            kubectl rollout status deployment/nginx \
            --timeout=120s

      - name: Show running pods
        run: KUBECONFIG=~/k8s-config kubectl get pods -o wide
```

::: {.fragment}

::: {.callout-note}
Triggered only when `k8s/**` files change — infra pushes (Terraform/Ansible) never trigger a workload deploy.
:::

:::


## ✅ Push the App and Verify

```bash
# 1. Copy k8s/ manifests into the repo and push → triggers deploy.yml
rsync -a lab/k8s-gitea-cicd/k8s/ ~/k8s-infra/k8s/
cd ~/k8s-infra
git add k8s/
git commit -m "feat: deploy nginx app"
git push gitea main
# → deploy.yml runs: kubectl apply → rollout → pods Running

# 2. Once the pipeline is green, curl the app
WORKER_IP=$(KUBECONFIG=~/k8s-config kubectl get nodes \
  -l '!node-role.kubernetes.io/control-plane' \
  -o jsonpath='{.items[0].status.addresses[0].address}')
curl http://${WORKER_IP}:30080
# Expected: nginx welcome page
```


## Rolling Update — GitOps Loop

```{mermaid}
sequenceDiagram
    participant Dev as Developer
    participant Git as Gitea
    participant Runner as act_runner (shell)
    participant K8s as Kubernetes

    Dev->>Git: git push (change image: nginx:1.27 → 1.28)
    Git->>Runner: trigger deploy.yml
    Runner->>K8s: kubectl apply -f k8s/
    K8s->>K8s: rolling update (new pod, old pod)
    K8s-->>Runner: rollout complete
    Runner-->>Git: job success
    Git-->>Dev: ✓ pipeline green

    Note over Dev,K8s: Rollback: git revert → push → kubectl apply
```

::: {.fragment}

::: {.callout-tip}
**Zero-downtime update** — Kubernetes replaces pods one at a time, ensuring at least `replicas - 1` pods remain available. No Ansible, no SSH, no downtime.
:::

:::

::: {.fragment}

**✅ Do it now — rolling update, rollback & scale**

```bash
cd ~/k8s-infra

# Update image → triggers deploy.yml
sed -i 's/nginx:1.27-alpine/nginx:1.28-alpine/' k8s/nginx-deployment.yaml
git add k8s/nginx-deployment.yaml
git commit -m "feat: upgrade nginx to 1.28"
git push gitea main

# Rollback via git revert
git revert HEAD --no-edit && git push gitea main

# Scale up
sed -i 's/replicas: 3/replicas: 5/' k8s/nginx-deployment.yaml
git add k8s/nginx-deployment.yaml
git commit -m "feat: scale nginx to 5 replicas"
git push gitea main
```

:::


# Monitoring & Performance {background-color="#0d1b2a" background-image="https://images.unsplash.com/photo-1551288049-bebda4e38f71?w=1600" background-size="cover" background-opacity="0.4"}

## Monitoring Stack Overview

::::{.columns}

:::{.fragment .column width="50%"}
```{mermaid}
%%{init: {'theme': 'dark'}}%%
graph TD
    K8S["Kubernetes<br/>Cluster"]
    MS["Metrics Server<br/>(kubectl top)"]
    PROM["Prometheus<br/>(time-series DB)"]
    GRAF["Grafana<br/>(dashboards)"]
    ALERT["Alertmanager<br/>(notifications)"]
    K8S --> MS
    K8S --> PROM
    PROM --> GRAF
    PROM --> ALERT

    style MS fill:#555,stroke:#333
    style PROM fill:#E74C3C,stroke:#a93226
    style GRAF fill:#27AE60,stroke:#1a7a42
    style ALERT fill:#7B68EE,stroke:#4b3fa0
    style K8S fill:#326CE5,stroke:#1a4a9f
```
:::

:::{.fragment .column width="50%"}
| Tool | Purpose | Install |
|---|---|---|
| **Metrics Server** | `kubectl top` (CPU/RAM) | manifest |
| **Prometheus** | Scrape & store metrics | Helm |
| **Grafana** | Visualise dashboards | Helm |
| **Alertmanager** | Route alerts | Helm |

*`kube-prometheus-stack` bundles all four in one Helm chart.*

```
k8s-gitea-cicd/
└── k8sadmin/
    ├── metrics-server/
    │   └── install.sh
    └── monitoring/
        ├── install.sh
        ├── cleanup.sh
        └── values.yaml
```
:::

::::

::: {.fragment}
::: {.callout-note}
**Two tiers:** Metrics Server serves the K8s scheduler and HPA. Prometheus gives deep, persistent, queryable metrics. Both are complementary — you need both.

Scripts live in `k8s-gitea-cicd/k8sadmin/` in the course repo — run them from the pipeline or directly on the control plane.
:::
:::

## Deploy Metrics Server

::::{.columns}

:::{.fragment .column width="50%"}
**Install** — `k8sadmin/metrics-server/install.sh`

```bash
# Official manifest (works on Multipass — note --kubelet-insecure-tls)
kubectl apply -f https://github.com/kubernetes-sigs/metrics-server/releases/latest/download/components.yaml

# Multipass clusters need the insecure-tls flag
kubectl patch deployment metrics-server \
  -n kube-system \
  --type=json \
  -p='[{"op":"add","path":"/spec/template/spec/containers/0/args/-",
       "value":"--kubelet-insecure-tls"}]'
```
:::

:::{.fragment .column width="50%"}
**Use**

```bash
# Node CPU & memory
kubectl top nodes

# Pod CPU & memory (all namespaces)
kubectl top pods -A

# Sort by CPU
kubectl top pods -A --sort-by=cpu

# Sort by memory
kubectl top pods -A --sort-by=memory
```
:::

::::

::: {.fragment}
::: {.callout-warning}
Without `--kubelet-insecure-tls`, Metrics Server fails on Multipass nodes because the node TLS cert doesn't include the VM's internal IP as a SAN.
:::
:::

## Deploy kube-prometheus-stack

::::{.columns}

:::{.fragment .column width="50%"}
**Install via Helm** — `k8sadmin/monitoring/install.sh`

```bash
helm repo add prometheus-community \
  https://prometheus-community.github.io/helm-charts
helm repo update

kubectl create namespace monitoring

helm upgrade --install kube-prom-stack \
  prometheus-community/kube-prometheus-stack \
  --namespace monitoring \
  -f k8sadmin/monitoring/values.yaml \
  --wait --timeout=10m
```
:::

:::{.fragment .column width="50%"}
**What gets deployed**

```bash
kubectl get pods -n monitoring

# NAME                                          READY
# alertmanager-kube-prom-stack-...              2/2
# kube-prom-stack-grafana-...                   3/3
# kube-prom-stack-kube-state-metrics-...        1/1
# kube-prom-stack-operator-...                  1/1
# kube-prom-stack-prometheus-node-exporter-...  1/1  # per node
# prometheus-kube-prom-stack-...                2/2
```

*~150 MB RAM total. Fits comfortably in a 3-node Multipass cluster.*
:::

::::

::: {.fragment}
::: {.callout-tip}
`kube-state-metrics` exports cluster-level metrics (Deployment replicas, Pod phases, Job completion). `node-exporter` exports host-level metrics (CPU, disk, network). Both are included automatically.
:::
:::

## Access Grafana Dashboards

::::{.columns}

:::{.fragment .column width="50%"}
**Port-forward to Grafana**

```bash
# Forward Grafana to localhost:3001
kubectl port-forward \
  svc/kube-prom-stack-grafana \
  3001:80 \
  -n monitoring

# Open in browser:
# http://localhost:3001
# user: admin  password: admin
```

**Or expose via NodePort**

```bash
kubectl patch svc kube-prom-stack-grafana \
  -n monitoring \
  -p '{"spec":{"type":"NodePort","ports":[{"port":80,"nodePort":32000}]}}'

# Access at http://<control-plane-ip>:32000
```
:::

:::{.fragment .column width="50%"}
**Recommended built-in dashboards**

| Dashboard ID | Name |
|---|---|
| **15757** | Node Exporter Full |
| **13770** | 1 Node Exporter |
| **6417** | Kubernetes Cluster |
| **8685** | K8s Capacity |
| **12740** | Kubernetes Monitoring |

```bash
# Import by ID in Grafana UI:
# Dashboards → Import → Enter ID → Load
```

*All dashboards auto-connect to the bundled Prometheus datasource.*
:::

::::

::: {.fragment}
::: {.callout-note}
Port 3001 is used to avoid colliding with Gitea on 3000. If both are running on the same host, adjust accordingly.
:::
:::

## ✅ Test: Verify Monitoring

::::{.columns}

:::{.fragment .column width="50%"}
**Metrics Server**

```bash
# Should show CPU/RAM for each node
kubectl top nodes

# Expected output:
# NAME           CPU(cores)   CPU%   MEMORY(bytes)   MEMORY%
# k8s-master     120m         6%     1200Mi          40%
# k8s-worker-1   80m          4%     900Mi           30%
# k8s-worker-2   75m          3%     850Mi           28%
```

```bash
# Should show all running pods
kubectl top pods -A --sort-by=cpu | head -15
```
:::

:::{.fragment .column width="50%"}
**Prometheus & Grafana**

```bash
# All monitoring pods Running
kubectl get pods -n monitoring

# Prometheus targets healthy
kubectl port-forward svc/kube-prom-stack-kube-promethe-prometheus \
  9090:9090 -n monitoring
# Open http://localhost:9090/targets
# All targets should be UP (green)

# Grafana reachable
kubectl port-forward svc/kube-prom-stack-grafana \
  3001:80 -n monitoring
# Open http://localhost:3001
# Login admin/admin → Dashboards → Browse
```
:::

::::

::: {.fragment}
::: {.callout-tip}
Run `kubectl top pods -n default` before and after deploying your nginx app to observe resource consumption changes in real time.
:::
:::

# `destroy.yml` — Teardown {background-color="#3a0a0a" background-image="https://images.unsplash.com/photo-1555680202-c86f0e12f086?w=1600" background-size="cover" background-opacity="1.0"}

## Controlled Teardown

```yaml
name: Destroy K8s Cluster
on:
  workflow_dispatch:   # manual only — never automatic

jobs:
  destroy:
    runs-on: self-hosted
    steps:
      - uses: actions/checkout@v4

      - name: Write SSH key pair
        run: |
          echo "${{ secrets.SSH_PRIVATE_KEY }}" > terraform/id_ed25519
          echo "${{ secrets.SSH_PUBLIC_KEY }}"  > terraform/id_ed25519.pub
          chmod 600 terraform/id_ed25519

      - name: tofu init
        run: tofu -chdir=terraform init -reconfigure

      - name: tofu destroy
        run: tofu -chdir=terraform destroy -auto-approve

      - name: Clean up host runner state
        run: |
          rm -f ~/k8s-config ~/k8s-hosts.ini
          rm -f ~/cloud-init.rendered.yml
          rm -f /tmp/k8s_join_command.sh
```

::: {.fragment}

::: {.callout-important}
**Always run destroy at the end of the lab.** Multipass VMs keep consuming RAM even when idle. `~/k8s-terraform.tfstate` persists on the host — `tofu destroy` needs it to know which VMs to delete.
:::

::: {.callout-warning}
`tofu init -reconfigure` is required because the runner's workspace is a fresh checkout — the `.terraform/` directory is never committed. Without it, OpenTofu cannot locate the backend and refuses to run.
:::

:::

## ✅ Trigger the Destroy

```bash
# In Gitea UI: k8s-infra → Actions → destroy.yml → Run workflow
# Or via API:
curl -s -X POST \
  http://localhost:3000/api/v1/repos/<user>/k8s-infra/actions/workflows/destroy.yml/dispatches \
  -H "Authorization: token <your-pat>" \
  -H "Content-Type: application/json" \
  -d '{"ref":"main"}'
```
